In [0]:
%run ../lib/email_sender_databricks

In [0]:
%run ../config/variables

In [0]:
from datetime import datetime
from databricks.sdk import WorkspaceClient

dbutils.widgets.text("parent_run_id", "", "Parent Run ID")

run_date = datetime.now().strftime('%Y-%m-%d')
run_id   = int(dbutils.widgets.get("parent_run_id"))

w     = WorkspaceClient()
run   = w.jobs.get_run(run_id=run_id)
tasks = run.tasks or []

rows = ""
for task in tasks:
    task_key   = task.task_key or "N/A"
    life_cycle = task.state.life_cycle_state.value if task.state and task.state.life_cycle_state else "N/A"
    result     = task.state.result_state.value      if task.state and task.state.result_state     else None

    if result == "SUCCESS":
        icon, label = "✅", "SUCCESS"
    elif result == "FAILED":
        icon, label = "❌", "FAILED"
    elif life_cycle == "SKIPPED":
        icon, label = "⏭️", "SKIPPED"
    else:
        icon, label = "⏳", life_cycle

    rows += f"""
        <tr>
            <td style="padding:8px; border:1px solid #ddd;">{task_key}</td>
            <td style="padding:8px; border:1px solid #ddd; text-align:center;">{icon} {label}</td>
        </tr>"""

html_body = f"""
<html>
<body style="font-family: Arial, sans-serif;">
    <p>Hi Team,</p>
    <p>PE Weekly ETL pipeline has <strong style="color:#dc3545;">FAILED</strong> on {run_date}. Please investigate immediately.</p>
    <h3>Task Summary:</h3>
    <table style="border-collapse:collapse; width:50%;">
        <tr style="background:#f2f2f2;">
            <th style="padding:8px; border:1px solid #ddd; text-align:left;">Task</th>
            <th style="padding:8px; border:1px solid #ddd; text-align:left;">Status</th>
        </tr>
        {rows}
    </table>
    <br>
    <p style="color:#666; font-size:12px;">Automated notification from Databricks PE Pipeline</p>
</body>
</html>
"""

send_custom_email(
    pipeline_name   = None,
    environment     = None,
    recipients      = pe_etl_recipients_started,
    subject         = f"🔴 PE Weekly ETL | FAILED | {run_date}",
    html_body       = html_body,
    is_html         = True,
    attachment_path = None
)


In [0]:
# ── Force job to show as FAILED in Databricks portal ─────────────────────────
dbutils.notebook.exit("PIPELINE_FAILED")